# DM Movement Heuristic Comparator
This notebook generates graphs and combines them into a latex file comparing movement heuristics under the baseline event treatment.

Heuristics are only used by trader types ZIDA and ZIDPA (and future trader type: ZIDTA).

Heuristics covered:
- **NONE**: Move until you make a contract; then stick forever. Note an issue: the movement flag is never rest regularly if using this heuristic. It only will reset (so can move again) when called to move randomly if movement_error_rate > 0.
- **PERIOD**: Reset movement flag every period. Effectively equivalent to ZID movement strategy.
- **WINDOW**: Reset movement flag if agent did not trade at least *n* times in the last *w* (window size) periods.
- **WEEK**: Reset movement flag at the start of every *week*.
- **START**: Reset movement flag every time agent.start() is called. Effectively every time the agents are initialized, changed and reinitialized.

In [ ]:
import random as rnd
import os
import sys

# Data processing
import numpy as np                              # import numpy
import pandas as pd

# Plotting
import matplotlib.pyplot as plt                 # import matplotlib

# Custom packages for simulation
# This works only if notebooks folder is in the "modules" parent folder
sys.path.insert(0, '..') # add modules folder (parent folder) into this notebook's path

# Environment modules
import environment.dm_agents as dm_agents

# Simulation modules
import simulations.dm_sim_experiment as exper

# Utility modules
import utils.nb_helpers as nb_help

# Latex module
import pylatex as ptx

import glob # file folder reader
import re # Regex module

# Agent type which uses movement rule heuristics
ZIDPA = dm_agents.ZIDPA

In [ ]:
# Functions to create the flag (movement heuristic) comparisons
def prepend_str(prepend_int):
    bs_str = str(prepend_int)
    if len(bs_str) == 1:
        r_str = "000" + bs_str
    elif len(bs_str) == 2:
        r_str = "00" + bs_str
    elif len(bs_str) == 3:
        r_str = "0" + bs_str
    else:
        r_str = bs_str
        
    return r_str

def compare_flags(num_weeks=100, movement_error_rate=0, compliance_rate=1, reset_freq=None, flag_specific=None, on_random=False, 
                 event_begin=48, event_end=52, title_prepend=None, folder="flag_reset_comparison", num_agents=20, trader_class_count=None,
                 min_agents=None, window_size=None, min_trades=None, num_periods=None, num_rounds=None, grid_size=None, num_units=None,lower_bound=None, upper_bound=None):
    
    if min_agents is None:
        min_agents = 2
    if window_size is None:
        window_size = 7
    if min_trades is None:
        min_trades = 1

    if reset_freq == "MIN_AGENTS" and flag_specific is not None:
        min_agents = flag_specific
    elif reset_freq == "WEEK" and flag_specific is not None:
        min_trades = flag_specific
    elif reset_freq == "WINDOW" and flag_specific is not None:
        window_size = flag_specific[0]
        min_trades = flag_specific[1]

    num_trials = 1
    if num_periods is None:
        num_periods = 7
    if num_rounds is None:
        num_rounds = 5
    if grid_size is None:
        grid_size = 15
    num_traders = num_agents
    if lower_bound is None:
        lower_bound = 200
    if upper_bound is None:
        upper_bound = 600
    if grid_size is None:
        grid_size = 15
    if num_units is None:
        num_units = 8
    
    if trader_class_count is None:
        trader_class_count =[(ZIDPA, num_agents)]

    common_controls = [event_begin, event_end,
                        num_rounds, grid_size,
                        num_traders, num_units,
                        lower_bound, upper_bound,
                        trader_class_count]

    tlt_append = ""
    fn_append = ""
    if movement_error_rate > 0:
        tlt_append = f"\nFlagOnRandom: {on_random}"
        fn_append = f"_fr_{str(on_random)[0]}"
    
    if reset_freq is None or reset_freq == "START" or reset_freq == "PERIOD":
        tlt = f"FlagRule: {reset_freq}.\nMoveErrorRate: {movement_error_rate}. ComplianceRate: {compliance_rate}." + tlt_append
        fn_1 = f"fr_{reset_freq}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"
    elif reset_freq == "MIN_AGENTS":
        tlt = f"FlagRule: {reset_freq}. MinAgents: {min_agents}.\nMoveErrorRate: {movement_error_rate}. ComplianceRate: {compliance_rate}." + tlt_append
        fn_1 = f"fr_{reset_freq}_ma_{min_agents}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_ma_{min_agents}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"
    elif reset_freq == "WEEK":
        tlt = f"FlagRule: {reset_freq}. MinTrades: {min_trades}.\nMoveErrorRate: {movement_error_rate}. ComplianceRate: {compliance_rate}." + tlt_append
        fn_1 = f"fr_{reset_freq}_mt_{min_trades}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_mt_{min_trades}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"
    elif reset_freq == "WINDOW":
        window_size = flag_specific[0]
        min_trades = flag_specific[1]
        tlt = f"FlagRule: {reset_freq}. MinTrades: {min_trades}. Window: {window_size}.\nMoveErrorRate: {movement_error_rate}. ComplianceRate: {compliance_rate}." + tlt_append
        fn_1 = f"fr_{reset_freq}_mt_{min_trades}_ws_{window_size}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_mt_{min_trades}_ws_{window_size}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"

    if title_prepend is not None:
        fn_1 = title_prepend + fn_1
        fn_10 = title_prepend + fn_10
    
    trial_1 = exper.make_event_monte_carlo("test1", num_trials, num_periods, num_weeks, *common_controls, movement_error_rate, compliance_rate, return_df=True, 
                                     reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents, reset_flag_on_random=on_random,
                                     reset_flag_window=window_size, reset_flag_min_trades=min_trades)
    plt.plot(trial_1['eff'])
    plt.axvspan(event_begin-0.5, event_end+0.5, color="grey", alpha=0.3)
    plt.title(tlt)
    plt.xlabel("Week")
    plt.ylabel("Efficiency")
    plt.savefig(folder+"/"+fn_1)
    plt.clf()

    num_trials = 10

    trial_10 = exper.make_event_monte_carlo("test10", num_trials, num_periods, num_weeks, *common_controls, movement_error_rate, compliance_rate, return_df=True, 
                                      reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents, reset_flag_on_random=on_random,
                                      reset_flag_window=window_size, reset_flag_min_trades=min_trades)
    
    td_boxplot = nb_help.format_df_for_boxplot(trial_10, 'week', 'eff')
    
    #plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)

    incr = 10
    if num_weeks == 500:
        incr = 40

    svn = folder+"/"+fn_10
    nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, incr), rbars=(event_begin-0.5, event_end+0.5), savename=svn)

def create_fname(num_weeks, movement_error_rate, compliance_rate, reset_freq, flag_specific, on_random, 
                                          event_begin, event_end, title_prepend, folder="flag_reset_comparison", min_agents=None, window_size=None,min_trades=None):
    if min_agents is None:
        min_agents = 2
    if window_size is None:
        window_size = 7
    if min_trades is None:
        min_trades = 1
        
    if reset_freq == "MIN_AGENTS" and flag_specific is not None:
        min_agents = flag_specific
    elif reset_freq == "WEEK" and flag_specific is not None:
        min_trades = flag_specific
    elif reset_freq == "WINDOW" and flag_specific is not None:
        window_size = flag_specific[0]
        min_trades = flag_specific[1]

    tlt_append = ""
    fn_append = ""
    if movement_error_rate > 0:
        tlt_append = f"\nFlagOnRandom: {on_random}"
        fn_append = f"_fr_{str(on_random)[0]}"
    
    if reset_freq is None or reset_freq == "START" or reset_freq == "PERIOD":
        fn_1 = f"fr_{reset_freq}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"
    elif reset_freq == "MIN_AGENTS":
        fn_1 = f"fr_{reset_freq}_ma_{min_agents}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_ma_{min_agents}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"
    elif reset_freq == "WEEK":
        fn_1 = f"fr_{reset_freq}_mt_{min_trades}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_mt_{min_trades}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"
    elif reset_freq == "WINDOW":
        window_size = flag_specific[0]
        min_trades = flag_specific[1]
        fn_1 = f"fr_{reset_freq}_mt_{min_trades}_ws_{window_size}_er_{movement_error_rate}_cr_{compliance_rate}_t1" + fn_append + ".png"
        fn_10 = f"fr_{reset_freq}_mt_{min_trades}_ws_{window_size}_er_{movement_error_rate}_cr_{compliance_rate}_t10" + fn_append + ".png"

    if title_prepend is not None:
        fn_1 = title_prepend + fn_1
        fn_10 = title_prepend + fn_10

    return fn_1, fn_10

# Generate cross-comparisons of the heuristics

In [9]:
# Treatment names
reset_freq_s = [None, "START", "PERIOD", "MIN_AGENTS", "WEEK", "WINDOW"]

movement_error_rate_s = [0, 0.0005] # [0, 0.05, 0.005, 0.0005]
compliance_rate_s = [0, 0.25, 0.5, 0.75, 1]

min_agents_s = [2]
on_random_s = [True] # [False, True]
window_size_s = [7, 14, 21]
min_trades_s = [1, 2, 3]

num_weeks_s = [100, 500, 500]
prep_int_s = [0, 1000, 2000]
event_begin_s = [48, 48, 48*5]
event_end_s = [52, 52, 52*5]

fname = "flag_reset_comparison_2"

for rf in reset_freq_s:
    for i in range(len(num_weeks_s)):
        prep_int = prep_int_s[i]
        n_wk = num_weeks_s[i]
        evnt_bgn = event_begin_s[i]
        evnt_end = event_end_s[i]
        if rf in [None, "START", "PERIOD", "MIN_AGENTS"]:
            for er in movement_error_rate_s:
                for cr in compliance_rate_s:
                    if er > 0:
                        for ors in on_random_s:
                            str_prep = prepend_str(prep_int)
                            compare_flags(num_weeks=n_wk, movement_error_rate=er, compliance_rate=cr, reset_freq=rf, flag_specific=None, on_random=ors, 
                                          event_begin=evnt_bgn, event_end=evnt_end, title_prepend=str_prep, folder=fname)
                            prep_int += 1
                    else:
                        str_prep = prepend_str(prep_int)
                        compare_flags(num_weeks=n_wk, movement_error_rate=er, compliance_rate=cr, reset_freq=rf, flag_specific=None, on_random=False, 
                            event_begin=evnt_bgn, event_end=evnt_end, title_prepend=str_prep, folder=fname)
                        prep_int += 1
                        
        if rf in ["WEEK"]:
            for er in movement_error_rate_s:
                for cr in compliance_rate_s:
                    for mt in min_trades_s:
                        if er > 0:
                            for ors in on_random_s:
                                str_prep = prepend_str(prep_int)
                                compare_flags(num_weeks=n_wk, movement_error_rate=er, compliance_rate=cr, reset_freq=rf, flag_specific=mt, on_random=ors, 
                                              event_begin=evnt_bgn, event_end=evnt_end, title_prepend=str_prep, folder=fname)
                                prep_int += 1
                        else:
                            str_prep = prepend_str(prep_int)
                            compare_flags(num_weeks=n_wk, movement_error_rate=er, compliance_rate=cr, reset_freq=rf, flag_specific=mt, on_random=False, 
                                event_begin=evnt_bgn, event_end=evnt_end, title_prepend=str_prep, folder=fname)
                            prep_int += 1
        
        if rf in ["WINDOW"]:
            for er in movement_error_rate_s:
                for cr in compliance_rate_s:
                    for mt in min_trades_s:
                        for ws in window_size_s:
                            fs = (ws, mt)
                            if er > 0:
                                for ors in on_random_s:
                                    str_prep = prepend_str(prep_int)
                                    compare_flags(num_weeks=n_wk, movement_error_rate=er, compliance_rate=cr, reset_freq=rf, flag_specific=fs, on_random=ors, 
                                                  event_begin=evnt_bgn, event_end=evnt_end, title_prepend=str_prep, folder=fname)
                                    prep_int += 1
                            else:
                                str_prep = prepend_str(prep_int)
                                compare_flags(num_weeks=n_wk, movement_error_rate=er, compliance_rate=cr, reset_freq=rf, flag_specific=fs, on_random=False, 
                                    event_begin=evnt_bgn, event_end=evnt_end, title_prepend=str_prep, folder=fname)
                                prep_int += 1
        
        


<Figure size 640x480 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 640x480 with 0 Axes>

...

<Figure size 800x800 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 800x800 with 0 Axes>

## Recover config from file names

In [ ]:
def parse_name(raw_file_name):
    
    num_periods = 7
    num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
    trader_objects = str([(ZIDPA, 20)])
    num_weeks = "SKIP"
    event_begin = "SKIP"
    event_end = "SKIP"
    
    image_name = raw_file_name.split('/')[1]
    image_name2 = re.sub('MIN_AGENTS', 'MINAGENTS', image_name) # fix my bad choice of using MAX_AGENTS instead of MAXAGENTS for some reason
    print(image_name2)
    inputs_split = image_name2.split('_')
    flag_rule = inputs_split[1]
    if flag_rule == "MINAGENTS": flag_rule = "MIN_AGENTS"
    
    # Made strange choices when choosing how to name these images - the length of the inputs is not stantardized across flags, only within - so need to use this map
    # 0000fr_None_er_0_cr_0.0005_t1_fr_T.png
    # 0005fr_PERIOD_er_0.0005_cr_0_t1_fr_T.png
    # 0021fr_START_er_0.0005_cr_0_t1_fr_T.png
    # 0023fr_MIN_AGENTS_ma_2_er_0.0005_cr_0_t1_fr_T.png -> mutated: 0023fr_MINAGENTS_ma_2_er_0.0005_cr_0_t1_fr_T.png
    # 0055fr_WEEK_mt_1_er_0.0005_cr_0_t1_fr_T.png
    # 0115fr_WINDOW_mt_1_ws_7_er_0.0005_cr_0_t1_fr_T.png
    standard_length_dict = {"None": 9, "PERIOD": 9, "START": 9, "MIN_AGENTS": 11, "WEEK": 11, "WINDOW": 13}
    standard_length = standard_length_dict[flag_rule]
    
    if len(inputs_split) < standard_length: # Made strange choice of not appending if "onRandom = T/F", only if Error Rate > 0 - so judge by length
        inputs_split.append('fr')
        inputs_split.append('T')
    
    trials = int(inputs_split[-3][1:].split('.')[0])
    movement_error_rate = float(inputs_split[-6])
    compliance_rate = float(inputs_split[-4])
    if flag_rule == "MIN_AGENTS":
        min_agents = int(inputs_split[3])
    else:
        min_agents = None
        
    if flag_rule == "WINDOW":
        window_size = int(inputs_split[5])
    else:
        window_size = None
        
    if flag_rule == "WEEK" or flag_rule == "WINDOW":
        min_trades = int(inputs_split[3])
    else:
        min_trades = None
        
    on_random = inputs_split[-1][0] == "T" # T indicates True - Flags Reset when random move
    prep_int = inputs_split[0]
    file_name = image_name
    
    cols_in = ['flag_rule', 'trials', 'num_weeks', 'num_periods', 'event_begin', 'event_end', 'num_rounds', 'grid_size', 'num_traders', 'trader_objects', 
      'num_units', 'lower_bound', 'upper_bound', 'movement_error_rate', 'compliance_rate', 'min_agents', 'on_random', 'window_size', 'min_trades',
      'prep_int', 'file_name']
    data_in = [[flag_rule, trials, num_weeks, num_periods, event_begin, event_end, num_rounds, grid_size, num_traders, trader_objects, num_units, lower_bound, upper_bound, movement_error_rate, compliance_rate,
              min_agents, on_random, window_size, min_trades, prep_int, file_name]]
    
    return pd.DataFrame(data=data_in, columns=cols_in)

In [55]:
name_frame = pd.DataFrame()
image_names = glob.glob(fname+'/*.png')
for img_f in image_names:
    name_frame = pd.concat([name_frame, parse_name(img_f)]).reset_index(drop=True)
name_frame.head()

2049fr_WINDOW_mt_2_ws_14_er_0.0005_cr_0_t10_fr_T.png
1000fr_WEEK_mt_1_er_0_cr_0_t1.png
0083fr_WINDOW_mt_1_ws_21_er_0.0005_cr_1_t1_fr_T.png
1065fr_WINDOW_mt_1_ws_21_er_0.0005_cr_0.5_t10_fr_T.png
2015fr_WINDOW_mt_3_ws_7_er_0_cr_0.25_t1.png
2003fr_None_er_0_cr_0.75_t10.png
0046fr_WINDOW_mt_1_ws_14_er_0.0005_cr_0_t10_fr_T.png
2007fr_START_er_0.0005_cr_0.5_t1_fr_T.png
2046fr_WINDOW_mt_1_ws_14_er_0.0005_cr_0_t1_fr_T.png
0005fr_None_er_0.0005_cr_0_t1_fr_T.png
1002fr_MINAGENTS_ma_2_er_0_cr_0.5_t10.png
0052fr_WINDOW_mt_3_ws_14_er_0.0005_cr_0_t1_fr_T.png
0009fr_START_er_0.0005_cr_1_t10_fr_T.png
2009fr_None_er_0.0005_cr_1_t1_fr_T.png
2070fr_WINDOW_mt_3_ws_14_er_0.0005_cr_0.5_t1_fr_T.png
0028fr_WINDOW_mt_1_ws_14_er_0_cr_0.75_t10.png
1004fr_PERIOD_er_0_cr_1_t1.png
1004fr_MINAGENTS_ma_2_er_0_cr_1_t1.png
2049fr_WINDOW_mt_2_ws_14_er_0.0005_cr_0_t1_fr_T.png
0006fr_MINAGENTS_ma_2_er_0.0005_cr_0.25_t1_fr_T.png
2040fr_WINDOW_mt_2_ws_14_er_0_cr_1_t1.png
2061fr_WINDOW_mt_3_ws_14_er_0.0005_cr_0.25_t1_fr_T.pn

,flag_rule,trials,num_weeks,num_periods,event_begin,event_end,num_rounds,grid_size,num_traders,trader_objects,...,lower_bound,upper_bound,movement_error_rate,compliance_rate,min_agents,on_random,window_size,min_trades,prep_int,file_name
0,WINDOW,10,SKIP,7,SKIP,SKIP,5,15,20,"[(<class 'environment.dm_agents.ZIDPA'>, 20)]",...,200,600,0.0005,0.00,None,True,14,2,2049fr,2049fr_WINDOW_mt_2_ws_14_er_0.0005_cr_0_t10_fr...
1,WEEK,1,SKIP,7,SKIP,SKIP,5,15,20,"[(<class 'environment.dm_agents.ZIDPA'>, 20)]",...,200,600,0.0000,0.00,None,True,None,1,1000fr,1000fr_WEEK_mt_1_er_0_cr_0_t1.png
2,WINDOW,1,SKIP,7,SKIP,SKIP,5,15,20,"[(<class 'environment.dm_agents.ZIDPA'>, 20)]",...,200,600,0.0005,1.00,None,True,21,1,0083fr,0083fr_WINDOW_mt_1_ws_21_er_0.0005_cr_1_t1_fr_...
3,WINDOW,10,SKIP,7,SKIP,SKIP,5,15,20,"[(<class 'environment.dm_agents.ZIDPA'>, 20)]",...,200,600,0.0005,0.50,None,True,21,1,1065fr,1065fr_WINDOW_mt_1_ws_21_er_0.0005_cr_0.5_t10_...
4,WINDOW,1,SKIP,7,SKIP,SKIP,5,15,20,"[(<class 'environment.dm_agents.ZIDPA'>, 20)]",...,200,600,0.0000,0.25,None,True,7,3,2015fr,2015fr_WINDOW_mt_3_ws_7_er_0_cr_0.25_t1.png


In [ ]:
# Generate dataframe mapping configuration to filename
cols_in = ['flag_rule', 'trials', 'num_weeks', 'num_periods', 'event_begin', 'event_end', 'num_rounds', 'grid_size', 'num_traders', 'trader_objects', 
          'num_units', 'lower_bound', 'upper_bound', 'movement_error_rate', 'compliance_rate', 'min_agents', 'on_random', 'window_size', 'min_trades',
          'prep_int', 'file_name']
data_in = []
ZIDPA = dm_agents.ZIDPA
num_periods = 7 # Constant
num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600 # Constants
trader_objects = [(ZIDPA, 20)]

for i in range(len(num_weeks_s)):
    prep_int = prep_int_s[i]
    n_wk = num_weeks_s[i]
    evnt_bgn = event_begin_s[i]
    evnt_end = event_end_s[i]
    for er in movement_error_rate_s:
        for cr in compliance_rate_s:
            for rf in [None, "START", "PERIOD", "MIN_AGENTS"]:
                if er > 0:
                    for ors in on_random_s:
                        str_prep = prepend_str(prep_int)
                        fn1, fn10 = create_fname(n_wk, er, cr, rf, 2, ors, 
                                          evnt_bgn, evnt_end, str_prep, folder=fname)
                        if rf == "MIN_AGENTS":
                            min_ag = min_agents_s[0]
                        else:
                            min_ag = None
                        data_in.append([rf, 1, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                            num_units, lower_bound, upper_bound, er, cr, min_ag, ors, None, None, prep_int, fn1])
                        data_in.append([rf, 10, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                            num_units, lower_bound, upper_bound, er, cr, min_ag, ors, None, None, prep_int, fn10])
                        prep_int += 1
                else:
                    str_prep = prepend_str(prep_int)
                    fn1, fn10 = create_fname(n_wk, er, cr, rf, 2, False,
                                    evnt_bgn, evnt_end, str_prep, folder=fname)
                    if rf == "MIN_AGENTS":
                        min_ag = min_agents_s[0]
                    else:
                        min_ag = None
                    data_in.append([rf, 1, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                        num_units, lower_bound, upper_bound, er, cr, min_ag, False, None, None, prep_int, fn1])
                    data_in.append([rf, 10, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                        num_units, lower_bound, upper_bound, er, cr, min_ag, False, None, None, prep_int, fn10])
                    prep_int += 1
                    
    for rf in ["WEEK"]:
        for er in movement_error_rate_s:
            for cr in compliance_rate_s:
                for mt in min_trades_s:
                    if er > 0:
                        for ors in on_random_s:
                            str_prep = prepend_str(prep_int)
                            fn1, fn10 = create_fname(n_wk, er, cr, rf, mt, ors, 
                                            evnt_bgn, evnt_end, str_prep, folder=fname)
                            data_in.append([rf, 1, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                                num_units, lower_bound, upper_bound, er, cr, None, ors, None, mt, prep_int, fn1])
                            data_in.append([rf, 10, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                                num_units, lower_bound, upper_bound, er, cr, None, ors, None, mt, prep_int, fn10])
                            prep_int += 1
                    else:
                        str_prep = prepend_str(prep_int)
                        fn1, fn10 = create_fname(n_wk, er, cr, rf, mt, False, 
                                        evnt_bgn, evnt_end, str_prep, folder=fname)
                        data_in.append([rf, 1, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                            num_units, lower_bound, upper_bound, er, cr, None, False, None, mt, prep_int, fn1])
                        data_in.append([rf, 10, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                            num_units, lower_bound, upper_bound, er, cr, None, False, None, mt, prep_int, fn10])
                        prep_int += 1
    
    for rf in ["WINDOW"]:
        for er in movement_error_rate_s:
            for cr in compliance_rate_s:
                for mt in min_trades_s:
                    for ws in window_size_s:
                        fs = (ws, mt)
                        if er > 0:
                            for ors in on_random_s:
                                str_prep = prepend_str(prep_int)
                                fn1, fn10 = create_fname(n_wk, er, cr, rf, fs, ors, 
                                            evnt_bgn, evnt_end, str_prep, folder=fname)
                                data_in.append([rf, 1, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                                    num_units, lower_bound, upper_bound, er, cr, None, ors, ws, mt, prep_int, fn1])
                                data_in.append([rf, 10, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                                    num_units, lower_bound, upper_bound, er, cr, None, ors, ws, mt, prep_int, fn10])
                                prep_int += 1
                        else:
                            str_prep = prepend_str(prep_int)
                            fn1, fn10 = create_fname(n_wk, er, cr, rf, fs, False, 
                                        evnt_bgn, evnt_end, str_prep, folder=fname)
                            data_in.append([rf, 1, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                                num_units, lower_bound, upper_bound, er, cr, None, False, ws, mt, prep_int, fn1])
                            data_in.append([rf, 10, n_wk, num_periods, evnt_bgn, evnt_end, num_rounds, grid_size, num_traders, str(trader_objects), 
                                num_units, lower_bound, upper_bound, er, cr, None, False, ws, mt, prep_int, fn10])
                            prep_int += 1

name_frame = pd.DataFrame(columns=cols_in, data=data_in)

## Images -> Latex

In [57]:
# Put the generated images into a LaTeX document

# Test the pylatex
# import pylatex

# image_filename = os.path.join(os.path.dirname(__file__), 'kitten.jpg')

nm_wk_s = num_weeks_s
evnt_strt_s = event_begin_s
evnt_end_s = event_end_s

doc = ptx.Document(default_filepath=os.path.join(fname, 'flag_comparator'))

name_frame2 = name_frame[(name_frame['on_random']==True)|(name_frame['movement_error_rate']==0)]
name_frame2['flag_rule'] = name_frame2['flag_rule'].apply(str)
name_frame2['file_name'] = name_frame2['file_name'].apply(str)
# Record common conditions
with doc.create(ptx.Section('Results of Different Flag Resetting Rules for Traded Flag (i.e. "should move" rules)')):
    doc.append('Number of Trials: 1 (line), 10 (boxplot)\n')
    doc.append('Number of Weeks: 100 or 500\n')
    doc.append('Event Period: 48 to 52, or 240 to 260\n')
    doc.append(f'Number of Periods (Week Length): {num_periods}\n')
    doc.append(f'Number of Trading Rounds (In Period): {num_rounds}\n')
    doc.append(f'Grid Size: {grid_size}\n')
    doc.append(f'Number of Traders: {num_traders}\n')
    doc.append(f'Number of Units: {num_units}\n')
    doc.append(f'Unit Valuation Range (uniform, symmetric): {lower_bound} to {upper_bound}\n')
    doc.append('Trader Types: ZIDPA (non-even), ZIDPR (event)\n')
    doc.append('Event Compliance (Transformation) Rates: [0, 0.25, 0.5, 0.75, 1]\n')
    doc.append('Reset Flag on Movement Error (Random Move): True\n')
    doc.append(ptx.NoEscape("\clearpage"))
    
    with doc.create(ptx.Subsection('Flag Reset Rule None - If Trade Once, Stay Forever')):
        name_frame2_n = name_frame2[name_frame2['flag_rule']=='None']
        for cr in compliance_rate_s:
            name_frame2_c = name_frame2_n[name_frame2_n['compliance_rate']==cr]
            for er in movement_error_rate_s:
                name_frame2_e = name_frame2_c[name_frame2_c['movement_error_rate']==er]
                for i in range(len(nm_wk_s)):
                    nm_wk = nm_wk_s[i]; evnt_strt = evnt_strt_s[i]; evnt_end = evnt_end_s[i]
                    img1 = name_frame2_e['file_name'].values[0+2*i]
                    img10 = name_frame2_e['file_name'].values[1+2*i]
                    with doc.create(ptx.Figure(position='!htb')) as figs:
                        with doc.create(ptx.SubFigure(
                                position='b',
                                width=ptx.NoEscape(r'0.45\linewidth'))) as left_fig:
                                    left_fig.add_image(img1, width=ptx.NoEscape(r'\linewidth'))
                                    left_fig.add_caption('One Trial')
                        with doc.create(ptx.SubFigure(
                                position='b',
                                width=ptx.NoEscape(r'0.45\linewidth'))) as right_fig:
                                    right_fig.append(ptx.StandAloneGraphic(
                                        image_options=ptx.base_classes.command.Options('clip', width=ptx.NoEscape(r'\linewidth'), trim='0 4cm 0 0'),
                                        filename=ptx.utils.fix_filename(img10)))
                                    right_fig.add_caption('10 Trials')
                        figs.add_caption(f"Figure Above: {nm_wk} Weeks, Event {evnt_strt}-{evnt_end}, Movement Error Rate {er}, Compliance Rate {cr}, Flag Rule: None")
            doc.append(ptx.NoEscape("\clearpage"))
            
    with doc.create(ptx.Subsection('Flag Reset Rule START - Each new week, start with traded=False')):
        name_frame2_n = name_frame2[name_frame2['flag_rule']=='START']
        for cr in compliance_rate_s:
            name_frame2_c = name_frame2_n[name_frame2_n['compliance_rate']==cr]
            for er in movement_error_rate_s:
                name_frame2_e = name_frame2_c[name_frame2_c['movement_error_rate']==er]
                for i in range(len(nm_wk_s)):
                    nm_wk = nm_wk_s[i]; evnt_strt = evnt_strt_s[i]; evnt_end = evnt_end_s[i]
                    name_frame2_e = name_frame2_n[name_frame2_n['movement_error_rate']==er]
                    img1 = name_frame2_e['file_name'].values[0+2*i]
                    img10 = name_frame2_e['file_name'].values[1+2*i]
                    with doc.create(ptx.Figure(position='!htb')) as figs:
                        with doc.create(ptx.SubFigure(
                                position='b',
                                width=ptx.NoEscape(r'0.45\linewidth'))) as left_fig:
                                    left_fig.add_image(img1, width=ptx.NoEscape(r'\linewidth'))
                                    left_fig.add_caption('One Trial')
                        with doc.create(ptx.SubFigure(
                                position='b',
                                width=ptx.NoEscape(r'0.45\linewidth'))) as right_fig:
                                    right_fig.append(ptx.StandAloneGraphic(
                                        image_options=ptx.base_classes.command.Options('clip', width=ptx.NoEscape(r'\linewidth'), trim='0 4cm 0 0'),
                                        filename=ptx.utils.fix_filename(img10)))
                                    right_fig.add_caption('10 Trials')
                        figs.add_caption(f"Figure Above: {nm_wk} Weeks, Event {evnt_strt}-{evnt_end}, Movement Error Rate {er}, Compliance Rate {cr}, Flag Rule: START")
                doc.append(ptx.NoEscape("\clearpage"))

    mn_agnt = min_agents_s[0]
    with doc.create(ptx.Subsection('Flag Reset Rule MIN_AGENTS - If less than minimium agents (2) at location, set traded=False')):
        name_frame2_n = name_frame2[name_frame2['flag_rule']=='MIN_AGENTS']
        for cr in compliance_rate_s:
            name_frame2_c = name_frame2_n[name_frame2_n['compliance_rate']==cr]
            for er in movement_error_rate_s:
                name_frame2_e = name_frame2_c[name_frame2_c['movement_error_rate']==er]
                for i in range(len(nm_wk_s)):
                    nm_wk = nm_wk_s[i]; evnt_strt = evnt_strt_s[i]; evnt_end = evnt_end_s[i]
                    name_frame2_e = name_frame2_n[name_frame2_n['movement_error_rate']==er]
                    img1 = name_frame2_e['file_name'].values[0+2*i]
                    img10 = name_frame2_e['file_name'].values[1+2*i]
                    with doc.create(ptx.Figure(position='!htb')) as figs:
                        with doc.create(ptx.SubFigure(
                                position='b',
                                width=ptx.NoEscape(r'0.45\linewidth'))) as left_fig:
                                    left_fig.add_image(img1, width=ptx.NoEscape(r'\linewidth'))
                                    left_fig.add_caption('One Trial')
                        with doc.create(ptx.SubFigure(
                                position='b',
                                width=ptx.NoEscape(r'0.45\linewidth'))) as right_fig:
                                    right_fig.append(ptx.StandAloneGraphic(
                                        image_options=ptx.base_classes.command.Options('clip', width=ptx.NoEscape(r'\linewidth'), trim='0 4cm 0 0'),
                                        filename=ptx.utils.fix_filename(img10)))
                                    right_fig.add_caption('10 Trials')
                        figs.add_caption(f"Figure Above: {nm_wk} Weeks, Event {evnt_strt}-{evnt_end}, Movement Error Rate {er}, Compliance Rate {cr}, Min Agents {mn_agnt}, Flag Rule: MIN_AGENTS")
                doc.append(ptx.NoEscape("\clearpage"))

        with doc.create(ptx.Subsection('Flag Reset Rule WEEK - Each new week, stay if reached min number of trades LAST week; record number of trades this week')):
            doc.append('Iterates over minimum of 1, 2, or 3 trades in One Week (7-periods, non-rolling)')
            name_frame2_n = name_frame2[name_frame2['flag_rule']=='WEEK']
            for mt in min_trades_s:
                name_frame2_g = name_frame2_n[name_frame2_n['min_trades']==mt]
                for cr in compliance_rate_s:
                    name_frame2_c = name_frame2_g[name_frame2_g['compliance_rate']==cr]
                    for er in movement_error_rate_s:
                        name_frame2_e = name_frame2_c[name_frame2_c['movement_error_rate']==er]
                        for i in range(len(nm_wk_s)):
                            nm_wk = nm_wk_s[i]; evnt_strt = evnt_strt_s[i]; evnt_end = evnt_end_s[i]
                            img1 = name_frame2_e['file_name'].values[0+2*i]
                            img10 = name_frame2_e['file_name'].values[1+2*i]
                            with doc.create(ptx.Figure(position='!htb')) as figs:
                                with doc.create(ptx.SubFigure(
                                        position='b',
                                        width=ptx.NoEscape(r'0.45\linewidth'))) as left_fig:
                                            left_fig.add_image(img1, width=ptx.NoEscape(r'\linewidth'))
                                            left_fig.add_caption('One Trial')
                                with doc.create(ptx.SubFigure(
                                        position='b',
                                        width=ptx.NoEscape(r'0.45\linewidth'))) as right_fig:
                                            right_fig.append(ptx.StandAloneGraphic(
                                                image_options=ptx.base_classes.command.Options('clip', width=ptx.NoEscape(r'\linewidth'), trim='0 4cm 0 0'),
                                                filename=ptx.utils.fix_filename(img10)))
                                            right_fig.add_caption('10 Trials')
                                figs.add_caption(f"Figure Above: {nm_wk} Weeks, Event {evnt_strt}-{evnt_end}, Movement Error Rate {er}, Compliance Rate {cr}, Min Trades {mt}, Flag Rule: WEEK")
                    doc.append(ptx.NoEscape("\clearpage"))

        with doc.create(ptx.Subsection('Flag Reset Rule WINDOW - Stay if reached min number of trades in a rolling window')):
            doc.append('Iterates over minimum of 1, 2, or 3 trades in each window. Window sizes: 7, 14, 21 periods.')
            name_frame2_n = name_frame2[name_frame2['flag_rule']=='WINDOW']
            for ws in window_size_s:
                name_frame2_d = name_frame2_n[name_frame2_n['window_size']==ws]
                with doc.create(ptx.Subsection(f'Window size {ws}')):
                    for mt in min_trades_s:
                        name_frame2_g = name_frame2_d[name_frame2_d['min_trades']==mt]
                        for cr in compliance_rate_s:
                            name_frame2_c = name_frame2_g[name_frame2_g['compliance_rate']==cr]
                            for er in movement_error_rate_s:
                                name_frame2_e = name_frame2_c[name_frame2_c['movement_error_rate']==er]
                                for i in range(len(nm_wk_s)):
                                    nm_wk = nm_wk_s[i]; evnt_strt = evnt_strt_s[i]; evnt_end = evnt_end_s[i]
                                    img1 = name_frame2_e['file_name'].values[0+2*i]
                                    img10 = name_frame2_e['file_name'].values[1+2*i]
                                    with doc.create(ptx.Figure(position='!htb')) as figs:
                                        with doc.create(ptx.SubFigure(
                                                position='b',
                                                width=ptx.NoEscape(r'0.45\linewidth'))) as left_fig:
                                                    left_fig.add_image(img1, width=ptx.NoEscape(r'\linewidth'))
                                                    left_fig.add_caption('One Trial')
                                        with doc.create(ptx.SubFigure(
                                                position='b',
                                                width=ptx.NoEscape(r'0.45\linewidth'))) as right_fig:
                                                    right_fig.append(ptx.StandAloneGraphic(
                                                        image_options=ptx.base_classes.command.Options('clip', width=ptx.NoEscape(r'\linewidth'), trim='0 4cm 0 0'),
                                                        filename=ptx.utils.fix_filename(img10)))
                                                    right_fig.add_caption('10 Trials')
                                        figs.add_caption(f"Figure Above: {nm_wk} Weeks, Event {evnt_strt}-{evnt_end}, Movement Error Rate {er}, Compliance Rate {cr}, Min Trades {mt}, Window Size {ws}")
                            doc.append(ptx.NoEscape("\clearpage"))
        
doc.generate_pdf(clean_tex=False)

# Display Examples of Heuristic Comparisons

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; 
lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="WINDOW"
min_agents=None
on_random=False
window_size = 14
min_trades = 2

trial_1 = exper.make_event_monte_carlo("test1", num_trials, num_periods, num_weeks, *common_controls, movement_error_rate, compliance_rate, return_df=True, 
                                 reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents, reset_flag_on_random=on_random,
                                 reset_flag_window=window_size, reset_flag_min_trades=min_trades)

plt.plot(trial_1['eff'])
plt.axvspan(event_begin-0.5, event_end+0.5, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="START"
min_agents=None
on_random=False

test1 = make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="PERIOD"
min_agents=None
on_random=False

test1 = make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="START"
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="PERIOD"
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=2
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=2
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.05
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.0005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.05
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.0005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.05
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.0005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

plt.plot(test1['eff'])
plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
plt.suptitle(f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}")
plt.title(f"ResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}")
plt.show()

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="START"
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="PERIOD"
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 40))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="START"
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 40))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq="PERIOD"
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 40))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=2
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0
compliance_rate = 1

reset_freq=None
min_agents=2
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 40))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.05
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.0005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=False

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.05
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 10; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0.0005
compliance_rate = 1

reset_freq=None
min_agents=None
on_random=True

test1 = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate, compliance_rate, return_df=True, reset_flag_frequency=reset_freq, reset_flag_min_agents=min_agents,
             reset_flag_on_random=on_random)

td_boxplot = nb_help.format_df_for_boxplot(test1, 'week', 'eff')

#plt.axvspan(event_begin, event_end, color="grey", alpha=0.3)
tlt = f"Move Error Rate: {movement_error_rate}; Compliance Rate: {compliance_rate}\nResetFreq: {reset_freq}; MinAgents: {min_agents}; OnRandom: {on_random}"
nb_help.plot_boxplot_data(td_boxplot, title=tlt, y_lab="Efficiency", x_lab="Week", x_ticks=np.arange(0, num_weeks+1, 10))

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 10; grid_size = 15; num_traders = 20; num_units = 8; 
lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0; compliance_rate = 1
treatment_vars = [movement_error_rate, compliance_rate]

test_rd = exper.make_event_monte_carlo("test1", *common_controls, *treatment_vars, return_df=True)

plt.plot(test_rd['eff'])

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 20; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 10; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0; compliance_rate = 1
treatment_vars = [movement_error_rate, compliance_rate]

test_pd = exper.make_event_monte_carlo("test_lg", *common_controls, *treatment_vars, return_df=True)

plt.plot(test_pd['eff'])

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 20; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 10; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0; compliance_rate = 1
treatment_vars = [movement_error_rate, compliance_rate]

test_lg = exper.make_event_monte_carlo("test_lg", *common_controls, *treatment_vars, return_df=True)

plt.plot(test_lg['eff'])

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 20; num_weeks = 500; event_begin = 90000; event_end = 9000000; num_rounds = 20; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0; compliance_rate = 1
treatment_vars = [movement_error_rate, compliance_rate]

test_sp = exper.make_event_monte_carlo("test_sp", *common_controls, *treatment_vars, return_df=True)

plt.plot(test_sp['eff'])

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 20; num_weeks = 500; event_begin = 90000; event_end = 9000000; num_rounds = 20; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0; compliance_rate = 1
treatment_vars = [movement_error_rate, compliance_rate]

test_sp_rd = exper.make_event_monte_carlo("test_sp_rd", *common_controls, *treatment_vars, return_df=True)

plt.plot(test_sp_rd['eff'])

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 20; num_weeks = 500; event_begin = 48; event_end = 52; num_rounds = 10; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

movement_error_rate = 0; compliance_rate = 1
treatment_vars = [movement_error_rate, compliance_rate]

test_pd = exper.make_event_monte_carlo("test_lg", *common_controls, *treatment_vars, return_df=True)

plt.plot(test_pd['eff'])

In [ ]:
test1, test1_p = exper.make_event_monte_carlo("test1", *common_controls, movement_error_rate = 0.0, compliance_rate = 1, return_df=True, return_period_df=True)
plotted_t1 = nb_help.collate_loc_plots(test1_p)

In [ ]:
mname = "Event_Sim_Test_1_E0_C1_v2_upd"
mname2 = mname+".mp4"
nb_help.movie_plotted(plotted_t1, movie_name=mname2, graph_folder=mname, title_val="Event Simulation - Error Rate=0, Compliance Rate=1", hue_val="Agents", size_val="Agents", scale="relative", hue_max=None, 
                  size_max=None, week_max=None, period_max=None, include_init=True, fps=1, subtitle=True)

In [ ]:
ZIDPA = dm_agents.ZIDPA
ZID = dm_agents.ZIDPR
num_trials = 1; num_periods = 7; num_weeks = 100; event_begin = 48; event_end = 52; num_rounds = 5; grid_size = 15; num_traders = 20; num_units = 8; lower_bound = 200; upper_bound = 600
trader_objects =[(ZIDPA, 20),(ZID, 0)]
common_controls = [num_trials, num_periods, num_weeks,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]

test2, test2_p = exper.make_event_monte_carlo("test2", *common_controls, movement_error_rate = 0.05, compliance_rate = 1, return_df=True, return_period_df=True)

In [ ]:
plt.plot(test2['eff'])

In [ ]:
plotted_t2 = nb_help.collate_loc_plots(test2_p)
nb_help.movie_plotted(plotted_t2, movie_name="Event_Sim_Test_1_E005_C1_upd.mp4", graph_folder="Event_Sim_E005_C1_upd", title_val="Event Simulation - Error Rate=0.05, Compliance Rate=1", hue_val="Agents", size_val="Agents", scale="relative", hue_max=None, 
                  size_max=None, week_max=None, period_max=None, include_init=True, fps=1, subtitle=True)

In [ ]:
test3, test3_p = exper.make_event_monte_carlo("test3", *common_controls, movement_error_rate = 0.005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test3['eff'])

In [ ]:
plotted_t3 = nb_help.collate_loc_plots(test3_p)
nb_help.movie_plotted(plotted_t3, movie_name="Event_Sim_Test_1_E0005_C1_upd.mp4", graph_folder="Event_Sim_E0005_C1_upd", title_val="Event Simulation - Error Rate=0.005, Compliance Rate=1", hue_val="Agents", size_val="Agents", scale="relative", hue_max=None, 
                  size_max=None, week_max=None, period_max=None, include_init=True, fps=1, subtitle=True)

In [ ]:
test4, test4_p = exper.make_event_monte_carlo("test4", *common_controls, movement_error_rate = 0.0005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test4['eff'])


In [ ]:
plotted_t4 = nb_help.collate_loc_plots(test4_p)
nb_help.movie_plotted(plotted_t4, movie_name="Event_Sim_Test_U_E00005_C1_upd.mp4", graph_folder="Event_Sim_E00005_C1_upd", title_val="Event Simulation - Error Rate=0.0005, Compliance Rate=1", hue_val="Agents", size_val="Agents", scale="relative", hue_max=None, 
                  size_max=None, week_max=None, period_max=None, include_init=True, fps=1, subtitle=True)

In [ ]:
test5, test5_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.00005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test5['eff'])

In [ ]:
plotted_t5 = nb_help.collate_loc_plots(test5_p)
nb_help.movie_plotted(plotted_t5, movie_name="Event_Sim_Test_U_E000005_C1_r_upd.mp4", graph_folder="Event_Sim_E000005_C1_r_upd", title_val="Event Simulation - Error Rate=0.0005, Compliance Rate=1", hue_val="Agents", size_val="Agents", scale="relative", hue_max=None, 
                  size_max=None, week_max=None, period_max=None, include_init=True, fps=1, subtitle=True)

In [ ]:
common_controls = [num_trials, num_periods, 500,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    20, num_units,
                    lower_bound, upper_bound,
                    [(ZIDPA, 20),(ZID, 0)]]
test7, test7_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.000005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test7['eff'])

In [ ]:
test6, test6_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.000005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test6['eff'])

In [ ]:
common_controls = [num_trials, num_periods, 500,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    6, num_units,
                    lower_bound, upper_bound,
                    [(ZIDPA, 6),(ZID, 0)]]
test7, test7_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.000005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test7['eff'])

In [ ]:
common_controls = [num_trials, num_periods, 500,
                    event_begin, event_end,
                    num_rounds, grid_size,
                    num_traders, num_units,
                    lower_bound, upper_bound,
                    trader_objects]
test8, test8_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.0005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test8['eff'])

In [ ]:
common_controls = [num_trials, num_periods, 100, # num_weeks
                    event_begin, event_end, 
                    num_rounds, grid_size,
                    6, num_units,
                    lower_bound, upper_bound,
                    [(ZIDPA, 6)]]
test9, test9_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.0005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test9['eff'])

In [ ]:
common_controls = [num_trials, num_periods, 100, # num_weeks
                    event_begin, event_end, 
                    num_rounds, grid_size,
                    20, num_units,
                    lower_bound, upper_bound,
                    [(ZIDPA, 20)]]
test9, test9_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.0005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test9['eff'])

In [ ]:
common_controls = [num_trials, num_periods, 100, # num_weeks
                    event_begin, event_end, 
                    num_rounds, grid_size,
                    50, num_units,
                    lower_bound, upper_bound,
                    [(ZIDPA, 50)]]
test9, test9_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.0005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test9['eff'])

In [ ]:
common_controls = [num_trials, num_periods, 500, # num_weeks
                    event_begin, event_end, 
                    num_rounds, grid_size,
                    50, num_units,
                    lower_bound, upper_bound,
                    [(ZIDPA, 50)]]
test9, test9_p = exper.make_event_monte_carlo("test5", *common_controls, movement_error_rate = 0.0005, compliance_rate = 1, return_df=True, return_period_df=True)
plt.plot(test9['eff'])